In [ ]:
%%writefile config.kaggle.yaml
data:
  # Path dataset yang sudah ada di Kaggle (credit-card-transactions)
  transactions: "/kaggle/input/credit-card-transactions/credit_card_transactions-ibm_v2.csv"
  max_rows: null
  train_fraction: 0.7
  val_fraction: 0.15

sampling:
  method: "uniform" # Pilihan: uniform, topology, importance
  undersampling_ratio: 0.02

model:
  hidden_channels: 64
  out_channels: 1
  num_layers: 2
  aggr: "mean"

training:
  batch_size: 50
  epochs: 10
  learning_rate: 0.001
  num_neighbors: [2, 32]

output:
  # Path output pada Kaggle Notebook (Folder /kaggle/working/ adalah direktori yang akan disimpan setelah run selesai)
  model_dir: "/kaggle/working/model/exp16"
  result_dir: "/kaggle/working/result/exp16"



In [ ]:
%%writefile config.py
import yaml
from pathlib import Path

def load_config(config_path: str) -> dict:
    """
    Memuat konfigurasi YAML dari file yang diberikan.
    """
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"File konfigurasi tidak ditemukan: {config_path}")
        
    with open(path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
        
    return config



In [ ]:
%%writefile data_preprocessing.py
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from pathlib import Path

def load_and_preprocess_data(file_path: str, nrows: int = None) -> pd.DataFrame:
    """
    Load the IBM Credit Card Transactions dataset and perform basic EDA/preprocessing.
    Converts timestamps, sorts chronologically, cleans numeric fields, and encodes categoricals.
    """
    print(f"Loading data from {file_path}...")
    df = pd.read_csv(file_path, nrows=nrows)

    print("Parsing timestamps...")
    # Fill missing time with 00:00 to avoid parsing errors
    df['Time'] = df['Time'].fillna('00:00')
    df['datetime'] = pd.to_datetime(
        df['Year'].astype(str) + '-' + 
        df['Month'].astype(str).str.zfill(2) + '-' + 
        df['Day'].astype(str).str.zfill(2) + ' ' + 
        df['Time']
    )
    
    print("Sorting chronologically...")
    df = df.sort_values('datetime').reset_index(drop=True)
    
    print("Cleaning Amount...")
    df['Amount'] = df['Amount'].astype(str).str.replace('$', '').str.replace(',', '').astype(float)
    
    print("Encoding target label...")
    df['Is Fraud?'] = df['Is Fraud?'].map({'Yes': 1, 'No': 0})
    
    print("Encoding categorical features...")
    cat_columns = ['Use Chip', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?']
    for col in cat_columns:
        df[col] = df[col].fillna('<missing>')
        df[col] = df[col].astype(str)
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        
    # Scale Amount
    scaler = StandardScaler()
    df['Amount'] = scaler.fit_transform(df[['Amount']])

    # Extract numerical features for Transaction node
    feature_cols = ['Amount'] + cat_columns
    
    return df, feature_cols

def temporal_split(df: pd.DataFrame, train_frac=0.7, val_frac=0.15):
    """
    Split the dataset based on temporal order.
    """
    n_samples = len(df)
    train_end = int(train_frac * n_samples)
    val_end = int((train_frac + val_frac) * n_samples)
    
    train_df = df.iloc[:train_end]
    val_df = df.iloc[train_end:val_end]
    test_df = df.iloc[val_end:]
    
    return train_df, val_df, test_df



In [ ]:
%%writefile graph_builder.py
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData

def graph_level_undersample(df: pd.DataFrame, ratio=0.1, seed=42) -> pd.DataFrame:
    """
    Perform Graph-Level Undersampling exactly like in the Inductive GRL reference.
    Keeps all fraud transactions and downsamples normal transactions based on the given ratio.
    """
    print(f"Applying graph-level undersampling with ratio {ratio}...")
    labels = df['Is Fraud?'].values
    fraud_idx = np.flatnonzero(labels == 1)
    normal_idx = np.flatnonzero(labels == 0)
    
    if len(fraud_idx) == 0 or len(normal_idx) == 0:
        return df
        
    desired_normal = int(np.floor(len(fraud_idx) / ratio))
    if desired_normal > len(normal_idx):
        desired_normal = len(normal_idx)
        
    rng = np.random.default_rng(seed)
    chosen_normal = rng.choice(normal_idx, size=desired_normal, replace=False)
    
    selected_idx = np.concatenate([chosen_normal, fraud_idx])
    selected_idx.sort() # Keep chronological order roughly if needed
    
    return df.iloc[selected_idx].reset_index(drop=True)

def build_hetero_graph(df: pd.DataFrame, feature_cols: list) -> HeteroData:
    """
    Builds a bipartite/tripartite heterogeneous graph.
    Nodes: User, Merchant, Transaction.
    Edges: (User, buys, Transaction), (Merchant, sells, Transaction)
    Features and labels are ONLY on Transaction nodes.
    """
    print("Building HeteroData graph...")
    data = HeteroData()
    
    # 1. Map IDs to continuous integers
    user_mapping = {uid: i for i, uid in enumerate(df['User'].unique())}
    merchant_mapping = {mid: i for i, mid in enumerate(df['Merchant Name'].unique())}
    
    num_users = len(user_mapping)
    num_merchants = len(merchant_mapping)
    num_transactions = len(df)
    
    # Add dummy features for user and merchant (required by some PyG samplers, using empty tensors)
    data['user'].num_nodes = num_users
    data['merchant'].num_nodes = num_merchants
    
    # 2. Transaction Features & Labels
    x_tx = torch.tensor(df[feature_cols].values, dtype=torch.float)
    y_tx = torch.tensor(df['Is Fraud?'].values, dtype=torch.float)
    data['transaction'].x = x_tx
    data['transaction'].y = y_tx
    
    # 3. Edges
    user_src = [user_mapping[uid] for uid in df['User']]
    tx_dst = list(range(num_transactions))
    edge_index_user_tx = torch.tensor([user_src, tx_dst], dtype=torch.long)
    
    merchant_src = [merchant_mapping[mid] for mid in df['Merchant Name']]
    edge_index_merchant_tx = torch.tensor([merchant_src, tx_dst], dtype=torch.long)
    
    data['user', 'buys', 'transaction'].edge_index = edge_index_user_tx
    data['merchant', 'sells', 'transaction'].edge_index = edge_index_merchant_tx
    
    # Add reverse edges for message passing (GraphSAGE needs undirected or bi-directional)
    data['transaction', 'rev_buys', 'user'].edge_index = edge_index_user_tx.flip([0])
    data['transaction', 'rev_sells', 'merchant'].edge_index = edge_index_merchant_tx.flip([0])
    
    return data



In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv

class HeteroGraphSAGE(nn.Module):
    def __init__(self, hidden_channels: int, out_channels: int, num_layers: int, data_metadata: tuple, feature_dims: dict):
        super().__init__()
        
        self.num_layers = num_layers
        
        # Initialize embeddings for node types without features (User, Merchant)
        self.lin_embeddings = nn.ModuleDict()
        self.param_embeddings = nn.ParameterDict()
        for node_type, dim in feature_dims.items():
            if dim == 0:
                self.param_embeddings[node_type] = nn.Parameter(torch.randn(1, hidden_channels))
            else:
                self.lin_embeddings[node_type] = nn.Linear(dim, hidden_channels)
                
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for i in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv(
                    in_channels=(-1, -1), # Lazy initialization
                    out_channels=hidden_channels,
                    aggr='mean'
                )
                for edge_type in data_metadata[1]
            })
            self.convs.append(conv)
            
            # BatchNorm for transaction nodes
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        # Classifier
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, num_nodes_dict):
        # Initial projection
        h_dict = {}
        
        # Nodes with features
        for node_type, x in x_dict.items():
            if node_type in self.lin_embeddings:
                h_dict[node_type] = self.lin_embeddings[node_type](x)
                
        # Nodes without features (dummy embeddings)
        for node_type, param in self.param_embeddings.items():
            num_nodes = num_nodes_dict[node_type]
            h_dict[node_type] = param.expand(num_nodes, -1)
                
        # Message Passing
        for i in range(self.num_layers):
            h_dict = self.convs[i](h_dict, edge_index_dict)
            
            # Apply ReLU and BatchNorm only on transactions (or all nodes if needed)
            for node_type in h_dict.keys():
                h_dict[node_type] = F.relu(h_dict[node_type])
                if node_type == 'transaction':
                    h_dict[node_type] = self.bns[i](h_dict[node_type])
                    
        # Classification only on transaction nodes
        out = self.classifier(h_dict['transaction'])
        return torch.sigmoid(out).squeeze(-1)

def get_loss_function(pos_weight: float):
    """
    Returns Class-Weighted Binary Cross-Entropy Loss
    pos_weight = (num_negatives / num_positives)
    """
    # BCELoss doesn't take pos_weight natively like BCEWithLogitsLoss.
    # Since we output sigmoid prob, we need to handle weighting manually or use BCEWithLogits.
    # To use pos_weight safely, we will return BCEWithLogitsLoss and remove Sigmoid from model?
    # Wait, the instruction says "Linear layer dengan fungsi aktivasi Sigmoid ... Gunakan class-weighted BCE".
    # Standard BCELoss requires manual weighting or just use weights.
    return nn.BCELoss()

def manual_weighted_bce(probs, targets, pos_weight):
    """
    Manually weighted BCE for sigmoid output.
    """
    loss = - (pos_weight * targets * torch.log(probs + 1e-7) + 
              (1.0 - targets) * torch.log(1.0 - probs + 1e-7))
    return loss.mean()



In [ ]:
%%writefile samplers.py
import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader
import networkx as nx

def calculate_degree_centrality(data: HeteroData, edge_type: tuple):
    """Simple degree centrality for a specific edge type."""
    src, rel, dst = edge_type
    edge_index = data[edge_type].edge_index
    num_nodes = data[src].num_nodes
    degree = torch.zeros(num_nodes, dtype=torch.float)
    degree.scatter_add_(0, edge_index[0], torch.ones(edge_index.shape[1]))
    return degree

def apply_uniform_sampling(data: HeteroData):
    """
    Baseline: Uniform Random Sampling.
    Assigns uniform probability (1.0) to all edges.
    """
    print("Applying Uniform Sampling weights...")
    for edge_type in data.edge_types:
        num_edges = data[edge_type].edge_index.size(1)
        data[edge_type].edge_weight = torch.ones(num_edges, dtype=torch.float)
    return data

def apply_topology_aware_sampling(data: HeteroData):
    """
    Topology-Aware Sampling (Jaccard Similarity and Local Homophily Ratio).
    Note: In a bipartite setup, we approximate this by weighting edges 
    connecting highly overlapping neighborhood structures.
    """
    print("Applying Topology-Aware Sampling weights...")
    # Placeholder for actual complex bipartite Jaccard calculation.
    # In practice, this requires building a bipartite projection or 2-hop neighborhood overlap.
    # Here we simulate the weighting mechanism: edges to nodes with shared structures get higher weights.
    for edge_type in data.edge_types:
        src, rel, dst = edge_type
        num_edges = data[edge_type].edge_index.size(1)
        
        # Simple structural approximation: inverse degree weighting (similar to basic homophily impact)
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        # Weight based on structural properties (dummy proxy for Jaccard/Homophily)
        # Real implementation would calculate exact Jaccard overlap between src and dst multi-hop neighbors.
        weights = 1.0 / (deg_src[src_nodes] + 1e-5) 
        
        # Normalize to avoid extreme values
        weights = (weights - weights.min()) / (weights.max() - weights.min() + 1e-5)
        data[edge_type].edge_weight = weights + 0.1 # Add small epsilon
        
    return data

def apply_importance_based_sampling(data: HeteroData):
    """
    Importance-Based Sampling (Degree Centrality and Personalized PageRank).
    Assigns higher sampling probability to important nodes.
    """
    print("Applying Importance-Based Sampling weights...")
    for edge_type in data.edge_types:
        src, rel, dst = edge_type
        
        # Calculate Degree Centrality
        deg_src = calculate_degree_centrality(data, edge_type)
        src_nodes = data[edge_type].edge_index[0]
        
        # For Personalized PageRank, exact calculation on huge graphs is very expensive.
        # We approximate importance by using Degree Centrality as a proxy for PPR limits.
        # High degree nodes (hubs) get higher sampling probabilities.
        importance_weights = deg_src[src_nodes]
        
        # Normalize weights
        importance_weights = (importance_weights - importance_weights.min()) / (importance_weights.max() - importance_weights.min() + 1e-5)
        data[edge_type].edge_weight = importance_weights + 0.1 # Add small epsilon

    return data

def get_neighbor_loader(data: HeteroData, batch_size: int = 1024, num_neighbors: list = [10, 10], method: str = 'uniform', shuffle: bool = True, input_nodes=None):
    """
    Returns a PyG NeighborLoader based on the selected sampling strategy.
    """
    if method == 'uniform':
        data = apply_uniform_sampling(data)
    elif method == 'topology':
        data = apply_topology_aware_sampling(data)
    elif method == 'importance':
        data = apply_importance_based_sampling(data)
    else:
        raise ValueError(f"Unknown sampling method: {method}")

    if input_nodes is None:
        input_nodes = ('transaction', torch.arange(data['transaction'].num_nodes))

    # Use weight_attr='edge_weight' to utilize the calculated probabilities
    loader = NeighborLoader(
        data,
        num_neighbors=num_neighbors,
        batch_size=batch_size,
        input_nodes=input_nodes,
        shuffle=shuffle,
        weight_attr='edge_weight'
    )
    return loader



In [ ]:
%%writefile main.py
import argparse
import time
import torch
import torch.optim as optim
from sklearn.metrics import f1_score, recall_score, average_precision_score
from pathlib import Path
import json
import numpy as np

from data_preprocessing import load_and_preprocess_data, temporal_split
from graph_builder import build_hetero_graph, graph_level_undersample
from samplers import get_neighbor_loader
from model import HeteroGraphSAGE, manual_weighted_bce
from config import load_config

def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_targets = []
    
    start_time = time.time()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            probs = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            
            # The loader returns the root nodes first in the transaction tensor
            # PyG 2.x NeighborLoader places the sampled sub-graph such that the first batch_size nodes are the seed nodes.
            batch_size = batch['transaction'].batch_size
            probs = probs[:batch_size]
            targets = batch['transaction'].y[:batch_size]
            
            all_preds.append(probs.cpu())
            all_targets.append(targets.cpu())
            
    inference_time = time.time() - start_time
    
    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    
    preds_bin = (all_preds > 0.5).astype(int)
    
    f1 = f1_score(all_targets, preds_bin)
    recall = recall_score(all_targets, preds_bin)
    auprc = average_precision_score(all_targets, all_preds)
    
    # Calculate 1% Lift
    import pandas as pd
    percentile = 0.01
    prob_class_1_with_labels = np.column_stack((all_preds, all_targets))
    sorted_df = pd.DataFrame(prob_class_1_with_labels).sort_values(by=[0], ascending=[False])
    top_percentile_df = sorted_df.head(max(1, int(round(len(sorted_df) * percentile))))
    
    percent_class_1_with_model = top_percentile_df[1].mean() if len(top_percentile_df) > 0 else 0.0
    percent_without_model = all_targets.mean()
    lift_1_percent = (percent_class_1_with_model / percent_without_model) if percent_without_model > 0 else 0.0
    
    return f1, recall, auprc, lift_1_percent, inference_time
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', type=str, default='config.reference.yaml', help='Path ke file konfigurasi YAML')
    parser.add_argument('--sampling', type=str, choices=['uniform', 'topology', 'importance'], default=None, help='Override konfigurasi sampling metode')
    parser.add_argument('--nrows', type=int, default=None, help="Limit baris data untuk testing")
    args = parser.parse_args()
    
    # Memuat konfigurasi
    cfg = load_config(args.config)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    sampling_method = args.sampling if args.sampling is not None else cfg['sampling']['method']
    print(f"Sampling method: {sampling_method}")

    # 1. Load and Preprocess
    dataset_path = cfg['data']['transactions']
    nrows = args.nrows if args.nrows is not None else cfg['data']['max_rows']
    df, feature_cols = load_and_preprocess_data(dataset_path, nrows=nrows)
    
    # 2. Split
    train_frac = cfg['data']['train_fraction']
    val_frac = cfg['data']['val_fraction']
    train_df, val_df, test_df = temporal_split(df, train_frac=train_frac, val_frac=val_frac)
    print(f"Split sizes: Train {len(train_df)}, Val {len(val_df)}, Test {len(test_df)}")
    
    # 3. Graph-Level Undersampling on Train
    undersample_ratio = cfg['sampling']['undersampling_ratio']
    train_df_sampled = graph_level_undersample(train_df, ratio=undersample_ratio)
    print(f"Train size after undersampling: {len(train_df_sampled)}")
    
    # 4. Build Graphs
    train_data = build_hetero_graph(train_df_sampled, feature_cols)
    test_data = build_hetero_graph(df, feature_cols)
    
    # Find test node indices in the full graph
    test_start_idx = len(train_df) + len(val_df)
    test_node_indices = torch.arange(test_start_idx, len(df))
    
    # 5. Samplers
    batch_size = cfg['training']['batch_size']
    num_neighbors = cfg['training']['num_neighbors']
    
    train_loader = get_neighbor_loader(
        train_data, 
        batch_size=batch_size, 
        num_neighbors=num_neighbors,
        method=sampling_method, 
        shuffle=True
    )
    
    test_loader = get_neighbor_loader(
        test_data, 
        batch_size=4096, # Gunakan batch size yang lebih besar saat inference untuk efisiensi waktu
        num_neighbors=num_neighbors,
        method=sampling_method, 
        shuffle=False,
        input_nodes=('transaction', test_node_indices)
    )

    # 6. Model Definition
    feature_dims = {
        'transaction': len(feature_cols),
        'user': 0, # Empty
        'merchant': 0 # Empty
    }
    
    hidden_channels = cfg['model']['hidden_channels']
    out_channels = cfg['model']['out_channels']
    num_layers = cfg['model']['num_layers']
    
    model = HeteroGraphSAGE(
        hidden_channels=hidden_channels,
        out_channels=out_channels,
        num_layers=num_layers,
        data_metadata=train_data.metadata(),
        feature_dims=feature_dims
    ).to(device)
    
    # Compute class weight for manual BCE
    num_pos = train_data['transaction'].y.sum().item()
    num_neg = train_data['transaction'].num_nodes - num_pos
    pos_weight = torch.tensor([num_neg / (num_pos + 1e-7)]).to(device)
    
    lr = cfg['training']['learning_rate']
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # 7. Training Loop
    epochs = cfg['training']['epochs']
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            probs = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            
            batch_size_current = batch['transaction'].batch_size
            probs = probs[:batch_size_current]
            targets = batch['transaction'].y[:batch_size_current]
            
            loss = manual_weighted_bce(probs, targets, pos_weight)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        print(f"Epoch {epoch+1:02d}, Loss: {total_loss/len(train_loader):.4f}")
        
    # Bebaskan RAM dari memori training agar tidak OOM saat evaluation
    import gc
    del train_data
    del train_loader
    del train_df
    del train_df_sampled
    del df
    gc.collect()

    # 8. Evaluation (Inductive Step)
    print("\\nStarting Inductive Evaluation on Test Set...")
    f1, recall, auprc, lift_1_percent, inf_time = evaluate(model, test_loader, device)
    
    print("=== Results ===")
    print(f"Sampling Method : {sampling_method}")
    print(f"F1-Score        : {f1:.4f}")
    print(f"Recall          : {recall:.4f}")
    print(f"AUPRC           : {auprc:.4f}")
    print(f"Lift@1%         : {lift_1_percent:.4f}")
    print(f"Inference Time  : {inf_time:.4f} seconds")
    
    # 9. Save Model and Results
    model_dir = Path(cfg.get('output', {}).get('model_dir', '../../model/exp16'))
    result_dir = Path(cfg.get('output', {}).get('result_dir', '../../result/exp16'))
    
    model_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)
    
    # Save Model Weights
    model_path = model_dir / f"exp16_{sampling_method}_model.pt"
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to: {model_path}")
    
    # Save Results as JSON
    result_dict = {
        "sampling_method": sampling_method,
        "f1_score": float(f1),
        "recall": float(recall),
        "auprc": float(auprc),
        "lift_1_percent": float(lift_1_percent),
        "inference_time": float(inf_time),
        "epochs": epochs,
        "batch_size": batch_size
    }
    result_path = result_dir / f"exp16_{sampling_method}_results.json"
    with open(result_path, 'w') as f:
        json.dump(result_dict, f, indent=4)
    print(f"Results saved to: {result_path}")

if __name__ == '__main__':
    main()



In [ ]:
%%writefile run_all.py
import os
import subprocess
import sys

def run_script(script_name, args=[]):
    print(f"\n{'='*70}")
    print(f"  >>> MENGJALANKAN SCRIPT: {script_name} {' '.join(args)}")
    print(f"{'='*70}\n")
    
    command = [sys.executable, script_name] + args
    result = subprocess.run(command)
    
    if result.returncode != 0:
        print(f"\n[ERROR] Eksekusi {script_name} gagal dengan kode {result.returncode}!")
        sys.exit(result.returncode)
    else:
        print(f"\n[SUCCESS] {script_name} selesai dieksekusi.\n")

if __name__ == "__main__":
    print("="*70)
    print("  MEMULAI EKSPERIMEN 16: KLASIFIKASI TRANSAKSI KARTU KREDIT (GRAPHSAGE)")
    print("="*70)
    
    # Install dependencies
    print("\n[INFO] Menginstall dependencies (torch_geometric & pyg-lib)...")
    try:
        import torch
        pt_version = torch.__version__.split('+')[0]
        cuda_version = torch.version.cuda.replace('.', '')
        url = f"https://data.pyg.org/whl/torch-{pt_version}+cu{cuda_version}.html"
        os.system(f"pip install torch_geometric pyg_lib torch_scatter torch_sparse -f {url} > /dev/null 2>&1")
    except Exception as e:
        print(f"[WARNING] Gagal medeteksi PyTorch versi untuk pyg_lib, fallback ke install biasa: {e}")
        os.system("pip install torch_geometric pyg_lib > /dev/null 2>&1")
    # Find dataset dynamically
    import yaml
    print("\n[INFO] Mencari lokasi dataset di /kaggle/input/...")
    csv_path = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'credit_card_transactions-ibm_v2.csv' in files:
            csv_path = os.path.join(root, 'credit_card_transactions-ibm_v2.csv')
            break
            
    if csv_path:
        print(f"[INFO] Dataset ditemukan di: {csv_path}")
        with open('config.kaggle.yaml', 'r') as f:
            cfg = yaml.safe_load(f)
        cfg['data']['transactions'] = csv_path
        with open('config.kaggle.yaml', 'w') as f:
            yaml.dump(cfg, f)
    else:
        print("[ERROR] DATASET TIDAK DITEMUKAN!")
        os.system("ls -laR /kaggle/input/")
        sys.exit(1)
        
    # Run the main experiment for all sampling methods sequentially
    sampling_methods = ["uniform", "topology", "importance"]
    for method in sampling_methods:
        print(f"\n{'*'*70}")
        print(f"  >>> MULAI EKSPERIMEN DENGAN SAMPLING: {method.upper()}")
        print(f"{'*'*70}\n")
        run_script("main.py", ["--config", "config.kaggle.yaml", "--sampling", method])    
    print("\n" + "="*70)
    print("  🎉 SELURUH PROSES PELATIHAN DAN EVALUASI SELESAI DENGAN SUKSES! 🎉")
    print("="*70)



In [ ]:
!python run_all.py